# SpriteForge Models: lưu model trên Kaggle (làm 1 lần)

Notebook này tải model từ HuggingFace vào **Output** của chính nó. Sau đó notebook SpriteForge gắn
Output này làm Input, nên **không phải tải lại mỗi session** và không phải upload từ máy bạn.

- Mặc định: FLUX.2 [klein] 4B cho engine "edit" (~11.6GB): model, bộ mã hoá chữ Qwen3 4B, VAE.
- `INCLUDE_POSE_SD = True`: thêm model engine cũ "pose" (~3.3GB): ControlNet OpenPose, IP-Adapter, CLIP-ViT-H.
- `INCLUDE_WAN = True`: thêm 3 file WAN 2.2 (~17GB) cho engine video. Output của Kaggle tối đa
  20GB: không bật cùng lúc với klein (chạy notebook này riêng một bản khác nếu cần WAN).

1. *Session options*: **Accelerator = None** (chỉ dùng CPU, không tốn quota GPU), **Internet = On**.
2. Bấm *Save Version* → **Save & Run All (Commit)** → chờ cho đến khi version chạy xong.
3. Mở notebook SpriteForge → **Add Input** → tab *Notebooks* (hoặc *Your Work*) → chọn notebook này.
   Đã gắn bản cũ (chỉ có WAN) thì chọn lại để dùng version mới nhất.

In [ ]:
import os, shutil
from huggingface_hub import get_hf_file_metadata, hf_hub_download, hf_hub_url

INCLUDE_KLEIN = True
INCLUDE_POSE_SD = False
INCLUDE_WAN = False
OUT = "/kaggle/working"   # Output của notebook, tối đa 20GB
WAN_REPO = "Comfy-Org/Wan_2.2_ComfyUI_Repackaged"
FILES = []  # (repo, đường dẫn trong repo, tên lưu)
if INCLUDE_KLEIN:
    FILES += [
        ("black-forest-labs/FLUX.2-klein-4b-fp8", "flux-2-klein-4b-fp8.safetensors",
         "flux-2-klein-4b-fp8.safetensors"),                                           # 3.8GB
        ("Comfy-Org/z_image_turbo", "split_files/text_encoders/qwen_3_4b.safetensors",
         "qwen_3_4b.safetensors"),                                                     # 7.5GB
        ("Comfy-Org/flux2-dev", "split_files/vae/flux2-vae.safetensors", "flux2-vae.safetensors")]
if INCLUDE_POSE_SD:
    FILES += [
        ("comfyanonymous/ControlNet-v1-1_fp16_safetensors", "control_v11p_sd15_openpose_fp16.safetensors",
         "control_v11p_sd15_openpose_fp16.safetensors"),                               # 0.7GB
        ("h94/IP-Adapter", "models/ip-adapter-plus_sd15.safetensors", "ip-adapter-plus_sd15.safetensors"),
        ("h94/IP-Adapter", "models/image_encoder/model.safetensors",
         "CLIP-ViT-H-14-laion2B-s32B-b79K.safetensors")]                               # 2.5GB
if INCLUDE_WAN:
    FILES += [(WAN_REPO, f"split_files/{p}", os.path.basename(p)) for p in (
        "diffusion_models/wan2.2_ti2v_5B_fp16.safetensors",      # 9.3GB
        "text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",  # 6.3GB
        "vae/wan2.2_vae.safetensors")]                            # 1.3GB
def expected_size(repo, path):
    try:
        return get_hf_file_metadata(hf_hub_url(repo, path)).size
    except Exception:
        return None

# Dọn model không còn dùng (VD 17GB WAN của version trước) để không vượt 20GB Output
wanted = {name for _, _, name in FILES}
for name in os.listdir(OUT):
    if name.endswith(".safetensors") and name not in wanted:
        print("Xoá file cũ không dùng:", name)
        os.remove(f"{OUT}/{name}")
print(f"Ổ Output còn trống: {shutil.disk_usage(OUT).free / 2**30:.1f} GB")

for repo, path, name in FILES:
    dst = f"{OUT}/{name}"
    if os.path.exists(dst):
        size = expected_size(repo, path)
        if size is None or os.path.getsize(dst) == size:
            print("Đã có:", name)
            continue
        print("File tải dở (sai dung lượng), tải lại:", name)
        os.remove(dst)
    print("Đang tải:", path)
    # Tải thẳng vào cùng ổ với Output -> chỉ đổi tên, không phải chép (không cần gấp đôi chỗ)
    os.replace(hf_hub_download(repo, path, local_dir=f"{OUT}/.hf"), dst)
shutil.rmtree(f"{OUT}/.hf", ignore_errors=True)

for name in sorted(os.listdir(OUT)):
    print(f"{os.path.getsize(f'{OUT}/{name}') / 2**30:6.2f} GB  {name}")